In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt



In [ ]:
wgms_data = pd.read_csv("/path/to/wgms/data/mass_balance.csv")
wgms_data = wgms_data.loc[wgms_data["glacier_name"] == "HINTEREIS F."]
wgms_data = wgms_data.loc[(wgms_data["begin_date"] >= "1999-01-01") & (wgms_data["begin_date"] <= "2010-01-01")]
keep_cols = ['year', 'time_system', 'begin_date', 'end_date', 'winter_balance',
       'winter_balance_unc', 'summer_balance', 'summer_balance_unc',
       'annual_balance', 'annual_balance_unc', 'remarks']
wgms_data = wgms_data[keep_cols]
wgms_data

In [ ]:
rgi_id = "RGI60-11.00897"
geod_ref = pd.read_csv("/path/to/geod_data/Hugonnet_21_MB/dh_11_rgi60_pergla_rates.csv")
geod_ref = geod_ref.loc[geod_ref['rgiid'] == rgi_id]
geod_ref = geod_ref.loc[geod_ref['period'] == "2000-01-01_2010-01-01"]
geod_ref = geod_ref[['dmdtda', 'err_dmdtda']]
geod_ref

In [ ]:
## klug et al. values
#Period BWGMS ref Bglac.hom ± σglac Bgeod.corr ± σgeod.corr 1B σcommon δ H095 β95
#2001/02 −0.647 +0.023 −0.624 ± 0.21 −0.685 ± 0.062 0.061 0.215 0.28 yes 94
#2002/03 −1.814 +0.018 −1.796 ± 0.21 −2.713 ± 0.183 0.917 0.276 3.33 no 9
#2003/04 −0.667 +0.016 −0.651 ± 0.21 −0.654 ± 0.063 0.003 0.216 0.01 yes 95
#2004/05 −1.061 +0.039 −1.022 ± 0.21 −1.028 ± 0.056 0.006 0.214 0.03 yes 95
#2005/06 −1.516 +0.023 −1.493 ± 0.21 −2.091 ± 0.100 0.598 0.229 2.61 no 26
#2006/07 −1.798 +0.015 −1.813 ± 0.21 −1.363 ± 0.041 −0.450 0.210 −2.14 no 43
#2007/08 −1.235 +0.011 −1.246 ± 0.21 −1.252 ± 0.046 0.006 0.211 0.03 yes 95
#2008/09 −1.182 +0.000 −1.182 ± 0.21 −1.209 ± 0.060 0.027 0.214 0.14 yes 95
#2009/10 −0.819 +0.027 −0.792 ± 0.21 −0.808 ± 0.029 0.016 0.208 0.08 yes 95
#2010/11 −1.420 +0.003 −1.423 ± 0.21 −1.249 ± 0.047 −0.174 0.211 −0.82 yes 87

klug_geod = pd.DataFrame({
    "year":     np.arange(2002, 2011+1),   # 2001/02->2002 ... 2010/11->2011
    "Bgeod":    [-0.685,-2.713,-0.654,-1.028,-2.091,-1.363,-1.252,-1.209,-0.808,-1.249],
    "Bgeod_unc":[ 0.062, 0.183, 0.063, 0.056, 0.100, 0.041, 0.046, 0.060, 0.029, 0.047],
})

In [ ]:
wgms = wgms_data[["year", "annual_balance", "annual_balance_unc"]].copy()
HUG = dict(dmdtda=-1.0425, err=0.261)

In [ ]:
wgms_win = wgms[(wgms.year >= 2000) & (wgms.year <= 2009)]
klug_win = klug_geod[(klug_geod.year >= 2002) & (klug_geod.year <= 2010)]
mean_wgms = wgms_win.annual_balance.mean()
mean_klug = klug_win.Bgeod.mean()

sem_wgms = wgms_win.annual_balance.std(ddof=1) / np.sqrt(len(wgms_win))
sem_klug = klug_win.Bgeod.std(ddof=1) / np.sqrt(len(klug_win))
 
print("=== Period-mean mass-balance rates (m w.e./yr) ===")
print(f"Hugonnet 2000-2010 (decadal geodetic): {HUG['dmdtda']:+.3f} +/- {HUG['err']:.3f}")
print(f"WGMS glaciological {wgms_win.year.min()}-{wgms_win.year.max()} (n={len(wgms_win)}): "
      f"{mean_wgms:+.3f}  (annual std {wgms_win.annual_balance.std():.3f})")
print(f"Klug geodetic {klug_win.year.min()}-{klug_win.year.max()} (n={len(klug_win)}): "
      f"{mean_klug:+.3f}  (annual std {klug_win.Bgeod.std():.3f})")
print(f"\nWithin Hugonnet +/-sigma band?  "
      f"WGMS: {abs(mean_wgms-HUG['dmdtda'])<=HUG['err']}   "
      f"Klug: {abs(mean_klug-HUG['dmdtda'])<=HUG['err']}")
print(f"Interannual std of WGMS ({wgms_win.annual_balance.std():.2f}) vs Hugonnet decadal "
      f"unc ({HUG['err']:.2f}) -> annual scatter >> decadal-mean uncertainty")
 
# ============================ plot ============================
plt.rcParams.update({'font.size': 22})
fig, ax = plt.subplots(figsize=(16, 9), dpi=300)
 
# Hugonnet decadal band + line
x0, x1 = 2000.25, 2010.25

# Shaded uncertainty band
ax.fill_between(
    [x0, x1],
    HUG['dmdtda'] - HUG['err'],
    HUG['dmdtda'] + HUG['err'],
    color='red',
    alpha=0.25,
    label=r'Hugonnet 2000-2010 ($-1.04 \pm 0.26$)',
    zorder=-1
)

# Mean line
ax.hlines(
    HUG['dmdtda'],
    x0,
    x1,
    color='red',
    lw=1.5,
    zorder=-1
)
 
# WGMS glaciological annual
ax.errorbar(wgms.year, wgms.annual_balance, yerr=wgms.annual_balance_unc,
            fmt='o-', color='#2E7D32', capsize=3, ms=6,
            label='WGMS', zorder=3)
# Klug geodetic annual
ax.errorbar(klug_geod.year, klug_geod.Bgeod, yerr=klug_geod.Bgeod_unc,
            fmt='s-', color='#1565C0', capsize=3, ms=6,
            label='Klug et al. 2018', zorder=3)
 
# period-mean lines + SEM shaded bands
wx0, wx1 = wgms_win.year.min(), wgms_win.year.max()
kx0, kx1 = klug_win.year.min(), klug_win.year.max()
ax.hlines(mean_wgms, wx0, wx1, color='#2E7D32', ls=':', lw=2,
          label=rf'WGMS mean {wx0}-{wx1} ({mean_wgms:+.2f}$\pm${sem_wgms:.2f})', zorder=2)
ax.fill_between([wx0, wx1], mean_wgms-sem_wgms, mean_wgms+sem_wgms, color='#2E7D32', alpha=0.12, zorder=1)
ax.hlines(mean_klug, kx0, kx1, color='#1565C0', ls=':', lw=2,
          label=rf'Klug geod. mean {kx0}-{kx1} ({mean_klug:+.2f}$\pm${sem_klug:.2f})', zorder=2)
ax.fill_between([kx0, kx1], mean_klug-sem_klug, mean_klug+sem_klug, color='#1565C0', alpha=0.12, zorder=1)
 
ax.set_xlabel('Hydrological Year')
ax.set_ylabel('Mass balance (m w.e. a$^{-1}$)')
ax.set_xticks(np.arange(2000, 2012))
ax.grid(True, alpha=0.3, zorder=-1)
ax.legend(fontsize=18, loc='lower right', framealpha=0.9)
#ax.set_title('Hintereisferner: geodetic vs glaciological mass balance')
fig.tight_layout()
plt.savefig("/path/to/figures/mb_comparison.png", bbox_inches="tight")

In [ ]:
## load albedo + snowline and check correlation
tsla = pd.read_csv("/path/to/snowlines/HEF-snowlines-1999-2010_manual_filtered.csv")
time_start_dt = pd.to_datetime("2000-01-01") #config starts with spinup - need to add 1year
time_end_dt = pd.to_datetime("2009-12-31")

tsla_true_obs = tsla.copy()
tsla_true_obs['LS_DATE'] = pd.to_datetime(tsla_true_obs['LS_DATE'])
print("Start date:", time_start_dt)
print("End date:", time_end_dt)
tsla_true_obs = tsla_true_obs.loc[(tsla_true_obs['LS_DATE'] > time_start_dt) & (tsla_true_obs['LS_DATE'] <= time_end_dt)]
tsla_true_obs.set_index('LS_DATE', inplace=True)
#Normalize standard deviation if necessary
tsla_true_obs['SC_stdev'] = (tsla_true_obs['SC_stdev']) / (tsla_true_obs['glacier_DEM_max'] - tsla_true_obs['glacier_DEM_min'])

thres_unc = (20) / (tsla_true_obs['glacier_DEM_max'].iloc[0] - tsla_true_obs['glacier_DEM_min'].iloc[0])
print(thres_unc)

## Set observational uncertainty where smaller to atleast model resolution (20m) and where larger keep it
sc_norm = np.where(tsla_true_obs['SC_stdev'] < thres_unc, thres_unc, tsla_true_obs['SC_stdev'])
tsla_true_obs['SC_stdev'] = sc_norm
tsla_true_obs

##
alb_obs_data = xr.open_dataset("/path/to/albedo/HEF_processed_HRZ-30CC-filter_albedos.nc")
#has nans where no glacier -> can build glacier-wide mean albedo for additional logp
alb_obs_data = alb_obs_data.sortby("time")
alb_obs_data

In [ ]:
common_times = pd.Index(tsla_true_obs.index).intersection(pd.Index(alb_obs_data.time.values))
df_subset = tsla_true_obs.loc[common_times]
ds_subset = alb_obs_data.sel(time=common_times)



In [ ]:
from scipy.stats import pearsonr

x = df_subset['TSL_normalized']
x_err = df_subset['SC_stdev']

# Extract xarray variables and convert to 1D numpy arrays
y = ds_subset['median_albedo'].values
y_err = ds_subset['sigma_albedo'].values

mask = ~np.isnan(x) & ~np.isnan(x_err) & ~np.isnan(y) & ~np.isnan(y_err)
x, x_err, y, y_err = x[mask], x_err[mask], y[mask], y_err[mask]

n_samples = len(x)

corr_coef, p_value = pearsonr(x, y)

plt.figure(figsize=(6, 6), dpi=300)
ax = plt.gca()
ax.set_aspect('equal', adjustable='box') # Ensures same scale/size for X and Y axes

plt.errorbar(
    x, y, 
    xerr=x_err, yerr=y_err, 
    fmt='o', ecolor='gray', elinewidth=0.8, capsize=1.5, 
    mfc='tab:blue', mec='white', mew=0.5, alpha=0.7, label='Data'
)

if n_samples > 1:
    slope, intercept = np.polyfit(x, y, 1)
    x_trend = np.linspace(x.min(), x.max(), 100)
    plt.plot(x_trend, slope * x_trend + intercept, color='red', linestyle='--', linewidth=1.5, label='Trendline')

plt.xlabel("TSL Normalized")
plt.ylabel("Median Albedo")
plt.grid(True, linestyle='--', alpha=0.5)

# Clean p-value formatting for extremely small values
if p_value < 1e-6:
    p_str = "< 1e-6"
else:
    p_str = f"= {p_value:.3f}"

stats_text = (
    f"r = {corr_coef:.2f}\n"
    f"p {p_str}\n"
    f"N = {n_samples}"
)

# Place text box in upper left (loc=2) or adjust coordinates manually
props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='lightgray')
ax.text(
    0.05, 0.95, stats_text, 
    transform=ax.transAxes, 
    fontsize=10, 
    verticalalignment='top', 
    bbox=props
)

plt.tight_layout()
plt.show()